In [1]:
import pymupdf
from pathlib import Path
from pprint import pprint

DOC_DIR = fr"../sample_docx"
DOC_PATH = Path(DOC_DIR, "digital.pdf")

pprint(f"Reading -> {DOC_PATH}")

doc = pymupdf.open(DOC_PATH)
pprint(doc.page_count)
text = ""   
for page in doc:
    print(page)
    text += page.get_text()

print(text)

'Reading -> ../sample_docx/digital.pdf'
1
page 0 of ../sample_docx/digital.pdf
 
 
PDF Test File 
 
Congratulations, your computer is equipped with a PDF (Portable Document Format) 
reader!  You should be able to view any of the PDF documents and forms available on 
our site.  PDF forms are indicated by these icons: 
  or  
.   
 
Yukon Department of Education 
Box 2703 
Whitehorse,Yukon 
Canada 
Y1A 2C6 
 
Please visit our website at:  http://www.education.gov.yk.ca/
   



In [2]:
import pymupdf, pytesseract, io
from pathlib import Path
from PIL import Image

DOC_PATH = Path("../sample_docx/Scanned-BG-1.pdf")

print(f"File exists: {DOC_PATH.exists()}")

doc = pymupdf.open(DOC_PATH)
print(f"Page count: {doc.page_count}")
text = ""
for i, page in enumerate(doc):
    pix = page.get_pixmap(dpi=300)
    img_bytes = pix.tobytes("png")
    img = Image.open(io.BytesIO(img_bytes))
    
    # Run Tesseract on the image
    text += f"\n{pytesseract.image_to_string(img)}\n"
print(text)

File exists: True
Page count: 1

BANK GUARANTEE

To

The Coastal Aquaculture Authority
Ministry of Agriculture

2™ Floor, Shastri Bhavan Annexe
26, Haddows Road

Chennai — 600 036

Dear Sir,

Subject: Performance Guarantee for Hatchery production of seed of SPF L. vannamei
granted by the Coastal Aquaculture Authority,Chennat.

This Deed of Guarantee executed by the (bank name) a Scheduled
Bank within the meaning of the Reserve Bank of India Act and carrying out banking business
including guarantee business and having its head office at (hereinafter referred to

as “the Bank”) in favour of Coastal Aquaculture Authority, Chennai having its office at the 2™
Floor, Shastri Bhavan Annexe, No.26, Haddows Road, Chennai-600006 (hereinafter referred to
as “Coastal Aquaculture Authority” for an amount not exceeding Rs.5,00,000/- (Rupees five lakh
only) at the request of (Supplier Name)
(hereinafter referred to as the “Supplier”).

This Guarantee is issued subject to the condition that the liabil

In [3]:
import traceback
from PIL import Image
import pymupdf, pytesseract, io

def get_ocr(file_path:str="")->dict:
    """
    Function which will get the raw text from the document and return a structured response

    Args:
        file_path (str, optional): File path from which we want to extract the raw text. Defaults to "".

    Returns:
        dict: 
        structure : {
        "text": <Extracted text>,
        "method": <Module of extraction>,
        "page_count": <Total pages in the document>,
        "status": < 1 : success, -1 : error in processing , 0 : processed but no extraction>
    }
    """
    text = ""
    result = {
        "text": "",
        "method": "",
        "page_count": 0,
        "status": -1
    }
    if not file_path:
        return {
            "text": "",
            "method": "NA",
            "page_count": 0,
            "status": -1
        }
    try:
        # trying extraction from pymupdf
        doc = pymupdf.open(file_path)
        result['page_count'] = doc.page_count
        for page in doc:
            text += page.get_text()
        
        if text:
            # if text extracted from the pymupdf libarary 
            result['text'] = text
            result['method'] = 'pymupdf'
            result['status'] = 1
        else:
            # trying to extract text from tesseract
            for i, page in enumerate(doc):
                pix = page.get_pixmap(dpi=300)
                img_bytes = pix.tobytes("png")
                img = Image.open(io.BytesIO(img_bytes))
                
                # Run Tesseract on the image
                text += f"\n{pytesseract.image_to_string(img)}\n"

            if text:
                result['text'] = text
                result['method'] = 'tesseract'
                result['status'] = 1
            else:
                result['text'] = text
                result['method'] = 'tesseract'
                result['status'] = 0

    except Exception as e:
        result['status'] = -1
        result['error'] = fr"{str(e)}\n{traceback.format_exc()}"
    return result

In [12]:
DOC_DIR = fr"../sample_docx"
digital_pdf = Path(DOC_DIR, "digital.pdf")
scanned_pdf = Path(DOC_DIR, "BG2_Security_Deposit_PWD.pdf")
text = get_ocr(file_path=scanned_pdf).get('text')

In [13]:
# Chunker logic
text = text.replace("\n", " ")

In [ ]:
char_start = 0
chunk_size = 500
chunk_overlap = 50
char_end = char_start + chunk_size
chunks:list = []
chunk_idx = 0
print(f"Len of text -> {len(text)}")
while not (char_start > len(text)):
    print(f"Char End -> {char_end}")
    print(f"Char Start -> {char_start}")
    chunks.append(
        {
            "chunk_id" : chunk_idx,
            "text" : text[char_start : char_end],
            "char_start" : char_start
        }
    )
    char_start = char_end - 50
    char_end = min(char_start + chunk_size, len(text))
    chunk_idx += 1
    print(f"Modified char start -> {char_start}")

import json
open(fr"./chunks.json","w").write(json.dumps(chunks, indent=4))

Len of text -> 3875
Char End -> 500
Char Start -> 0
Modified char start -> 450
Char End -> 950
Char Start -> 450
Modified char start -> 900
Char End -> 1400
Char Start -> 900
Modified char start -> 1350
Char End -> 1850
Char Start -> 1350
Modified char start -> 1800
Char End -> 2300
Char Start -> 1800
Modified char start -> 2250
Char End -> 2750
Char Start -> 2250
Modified char start -> 2700
Char End -> 3200
Char Start -> 2700
Modified char start -> 3150
Char End -> 3650
Char Start -> 3150
Modified char start -> 3600
Char End -> 4100
Char Start -> 3600
Modified char start -> 4050


5057